<a href="https://colab.research.google.com/github/malikasadnadir-max/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/malikasadnadir-max/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [14]:
# ML-07 SETUP — Clone the FlyRank repository

import os
from pathlib import Path

repo_path = Path("/content/flyrank-ml-internship")

if not repo_path.exists():
    !git clone https://github.com/malikasadnadir-max/flyrank-ml-internship.git

print("Repository exists:", repo_path.exists())
print("Repository path:", repo_path)

Repository exists: True
Repository path: /content/flyrank-ml-internship


In [15]:
# Check that the required dataset exists

from pathlib import Path

data_path = Path(
    "/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"
)

print("Dataset exists:", data_path.exists())

if data_path.exists():
    print("Dataset path:", data_path)
    print("Dataset size (MB):", round(data_path.stat().st_size / (1024**2), 2))
else:
    print("Dataset was not found.")

Dataset exists: True
Dataset path: /content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv
Dataset size (MB): 6.42


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### My rule and its reason codes

I will build a simple baseline rule using two observable signals: **staleness** and **CTR relative to search position**.

I checked both signals using bucket tables with the number of observations (`n`) and the observed declining rate. Staleness is linked to FlyRank's refresh-flag logic, while CTR relative to position is linked to CTR-fix reasoning.

The staleness check was **CONFIRMED**: the 365+ day bucket had a 60.00% declining rate compared with 51.20% for the <90 day bucket, a difference of 8.80 percentage points.

The CTR-versus-position check was also **CONFIRMED**: pages with CTR below 0.5% and position <=20 had a 62.71% declining rate compared with 47.53% for pages with CTR >=0.5% and position <=20, a difference of 15.18 percentage points.

Based on these observed differences, my rule will prioritize pages that have sufficient search visibility and show one of these observable opportunities.

Each page will receive one reason code:

* `STALE` — the page has not been updated recently enough and has sufficient search visibility.
* `LOW_CTR_POSITION` — the page has sufficient impressions, ranks within the first 20 positions, and has low CTR.
* `NO_CLEAR_SIGNAL` — neither condition is strong enough to trigger an action.

The action labels will be `REFRESH`, `CTR_REVIEW`, or `MONITOR`.

The score is a transparent decision-support ranking. It is not a prediction of future performance.


In [16]:
# SECTION 1 — Load data and check the two signals

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# Load the dataset
# ------------------------------------------------------------

repo_root = Path("/content/flyrank-ml-internship")

data_path = repo_root / "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(data_path)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

# ------------------------------------------------------------
# Check required columns
# ------------------------------------------------------------

required_columns = [
    "days_since_last_update",
    "impressions_90d",
    "ctr",
    "avg_position",
    "trend_direction",
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

# Convert numeric fields safely
for col in [
    "days_since_last_update",
    "impressions_90d",
    "ctr",
    "avg_position",
]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# ------------------------------------------------------------
# Label is used ONLY for checking whether the signals are real.
# It will NOT be used by the baseline score.
# ------------------------------------------------------------

df["is_declining_label"] = (
    df["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)

# ============================================================
# SIGNAL 1 — STALENESS
# ============================================================

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-np.inf, 89, 179, 364, np.inf],
    labels=[
        "<90 days",
        "90-179 days",
        "180-364 days",
        "365+ days",
    ],
)

staleness_table = (
    df.groupby(
        "staleness_bucket",
        observed=False
    )
    .agg(
        n=("is_declining_label", "size"),
        declining_rate=("is_declining_label", "mean"),
    )
    .reset_index()
)

staleness_table["declining_rate"] = (
    staleness_table["declining_rate"] * 100
).round(2)

print("\n" + "=" * 70)
print("SIGNAL 1 — STALENESS")
print("=" * 70)

print(staleness_table.to_string(index=False))

# Compare oldest and youngest buckets
oldest_rate = staleness_table.loc[
    staleness_table["staleness_bucket"] == "365+ days",
    "declining_rate"
].iloc[0]

youngest_rate = staleness_table.loc[
    staleness_table["staleness_bucket"] == "<90 days",
    "declining_rate"
].iloc[0]

staleness_difference = oldest_rate - youngest_rate

if staleness_difference >= 5:
    staleness_verdict = "CONFIRMED"
elif staleness_difference <= -5:
    staleness_verdict = "OPPOSITE"
else:
    staleness_verdict = "MIXED"

print("\nStaleness verdict:", staleness_verdict)
print(
    "Oldest minus youngest declining rate:",
    round(staleness_difference, 2),
    "percentage points"
)

# ============================================================
# SIGNAL 2 — CTR RELATIVE TO POSITION
# ============================================================

# Require at least 500 impressions and a valid position.
eligible = (
    (df["impressions_90d"] >= 500)
    & (df["avg_position"] > 0)
    & (df["avg_position"] <= 20)
)

df["ctr_position_bucket"] = np.select(
    [
        eligible & (df["ctr"] < 0.5),
        eligible & (df["ctr"] >= 0.5),
    ],
    [
        "Low CTR (<0.5%) with position <=20",
        "CTR >=0.5% with position <=20",
    ],
    default="Outside signal scope",
)

ctr_position_table = (
    df.groupby(
        "ctr_position_bucket"
    )
    .agg(
        n=("is_declining_label", "size"),
        declining_rate=("is_declining_label", "mean"),
    )
    .reset_index()
)

ctr_position_table["declining_rate"] = (
    ctr_position_table["declining_rate"] * 100
).round(2)

print("\n" + "=" * 70)
print("SIGNAL 2 — CTR RELATIVE TO POSITION")
print("=" * 70)

print(ctr_position_table.to_string(index=False))

# Compare low CTR with higher CTR within positions <=20
low_ctr_rate = ctr_position_table.loc[
    ctr_position_table["ctr_position_bucket"]
    == "Low CTR (<0.5%) with position <=20",
    "declining_rate"
].iloc[0]

normal_ctr_rate = ctr_position_table.loc[
    ctr_position_table["ctr_position_bucket"]
    == "CTR >=0.5% with position <=20",
    "declining_rate"
].iloc[0]

ctr_difference = low_ctr_rate - normal_ctr_rate

if ctr_difference >= 5:
    ctr_verdict = "CONFIRMED"
elif ctr_difference <= -5:
    ctr_verdict = "OPPOSITE"
else:
    ctr_verdict = "MIXED"

print("\nCTR-position verdict:", ctr_verdict)
print(
    "Low-CTR minus higher-CTR declining rate:",
    round(ctr_difference, 2),
    "percentage points"
)

# ============================================================
# FINAL VERDICTS
# ============================================================

print("\n" + "=" * 70)
print("FINAL SIGNAL VERDICTS")
print("=" * 70)

print("Staleness:", staleness_verdict)
print("CTR vs position:", ctr_verdict)

Dataset loaded successfully.
Shape: (30000, 44)

SIGNAL 1 — STALENESS
staleness_bucket     n  declining_rate
        <90 days 20655           51.20
     90-179 days  9171           61.11
    180-364 days   169           46.75
       365+ days     5           60.00

Staleness verdict: CONFIRMED
Oldest minus youngest declining rate: 8.8 percentage points

SIGNAL 2 — CTR RELATIVE TO POSITION
               ctr_position_bucket     n  declining_rate
     CTR >=0.5% with position <=20  2264           47.53
Low CTR (<0.5%) with position <=20  9759           62.71
              Outside signal scope 17977           50.43

CTR-position verdict: CONFIRMED
Low-CTR minus higher-CTR declining rate: 15.18 percentage points

FINAL SIGNAL VERDICTS
Staleness: CONFIRMED
CTR vs position: CONFIRMED


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### Build the ranked queue

I will encode one transparent rule using the two confirmed signals.

A page receives a higher score when it is sufficiently stale and has enough search visibility, or when it has enough search visibility, ranks within the first 20 positions, and has CTR below 0.5%.

Each page receives exactly one reason code and one action label. The score is used only to rank pages for review and is not a prediction of future performance.

The ranked queue will be written to `work/outputs/baseline_action_score.csv`.


In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# SECTION 2 — Build the ranked queue

# ------------------------------------------------------------
# Rule conditions
# ------------------------------------------------------------

stale_condition = (
    (df["days_since_last_update"] >= 180)
    & (df["impressions_90d"] >= 500)
)

low_ctr_position_condition = (
    (df["impressions_90d"] >= 500)
    & (df["avg_position"] > 0)
    & (df["avg_position"] <= 20)
    & (df["ctr"] < 0.5)
)

# ------------------------------------------------------------
# Transparent baseline score
# ------------------------------------------------------------

df["baseline_score"] = (
    stale_condition.astype(int) * 2
    + low_ctr_position_condition.astype(int)
)

# ------------------------------------------------------------
# ONE reason code per row
# ------------------------------------------------------------

df["reason_code"] = np.select(
    [
        stale_condition,
        low_ctr_position_condition,
    ],
    [
        "STALE",
        "LOW_CTR_POSITION",
    ],
    default="NO_CLEAR_SIGNAL",
)

# ------------------------------------------------------------
# Action label
# ------------------------------------------------------------

df["action"] = np.select(
    [
        df["reason_code"].eq("STALE"),
        df["reason_code"].eq("LOW_CTR_POSITION"),
    ],
    [
        "REFRESH",
        "CTR_REVIEW",
    ],
    default="MONITOR",
)

# ------------------------------------------------------------
# Rank all pages
# ------------------------------------------------------------

df["baseline_rank"] = (
    df["baseline_score"]
    .rank(method="first", ascending=False)
    .astype(int)
)

ranked_queue = (
    df.sort_values("baseline_rank")
      .reset_index(drop=True)
)

# ------------------------------------------------------------
# Write required CSV
# ------------------------------------------------------------

output_path = repo_root / "work/outputs/baseline_action_score.csv"

output_path.parent.mkdir(parents=True, exist_ok=True)

output_columns = [
    "content_id",
    "client_id",
    "baseline_rank",
    "baseline_score",
    "reason_code",
    "action",
    "impressions_90d",
    "ctr",
    "avg_position",
    "days_since_last_update",
    "content_age_days",
]

ranked_queue[output_columns].to_csv(
    output_path,
    index=False
)

# ------------------------------------------------------------
# Checks
# ------------------------------------------------------------

print("=" * 70)
print("BASELINE QUEUE CREATED")
print("=" * 70)

print("Rows ranked:", len(ranked_queue))
print("Output:", output_path)

print("\nScore distribution:")
print(
    ranked_queue["baseline_score"]
    .value_counts()
    .sort_index(ascending=False)
)

print("\nReason-code distribution:")
print(
    ranked_queue["reason_code"]
    .value_counts()
)

print("\nAction distribution:")
print(
    ranked_queue["action"]
    .value_counts()
)

print("\nTop 10:")
print(
    ranked_queue[
        [
            "baseline_rank",
            "baseline_score",
            "reason_code",
            "action",
            "impressions_90d",
            "ctr",
            "avg_position",
            "days_since_last_update",
        ]
    ]
    .head(10)
    .to_string(index=False)
)

BASELINE QUEUE CREATED
Rows ranked: 30000
Output: /content/flyrank-ml-internship/work/outputs/baseline_action_score.csv

Score distribution:
baseline_score
3       10
2        7
1     9749
0    20234
Name: count, dtype: int64

Reason-code distribution:
reason_code
NO_CLEAR_SIGNAL     20234
LOW_CTR_POSITION     9749
STALE                  17
Name: count, dtype: int64

Action distribution:
action
MONITOR       20234
CTR_REVIEW     9749
REFRESH          17
Name: count, dtype: int64

Top 10:
 baseline_rank  baseline_score reason_code  action  impressions_90d  ctr  avg_position  days_since_last_update
             1               3       STALE REFRESH             4556 0.33          16.4                     194
             2               3       STALE REFRESH              821 0.24           5.8                     301
             3               3       STALE REFRESH              545 0.18          17.8                     183
             4               3       STALE REFRESH             

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-10 review

I reviewed the top ten pages produced by the baseline ranking.

All ten received the `STALE` reason code and `REFRESH` action because they met the stale-page condition and also had sufficient search visibility.

The ranking is directional rather than predictive. A high-ranked page could still be a weak pick if the page was recently reviewed for reasons not represented in the available data, if its current performance does not justify a refresh, or if the observed CTR and position do not reflect the full search context.

For each row, the review records the action, why the page was selected, a confidence note, and what could make the rule wrong.


In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# SECTION 3 — Top-10 review

top10 = ranked_queue.head(10).copy()

def get_confidence_note(row):
    if row["baseline_score"] == 3:
        return (
            "Higher confidence in rule fit: the page satisfies "
            "both observable conditions."
        )
    elif row["baseline_score"] == 2:
        return (
            "Moderate confidence in rule fit: the page satisfies "
            "the staleness condition."
        )
    else:
        return (
            "Lower confidence in rule fit: the page has only "
            "one weaker observable condition."
        )


def get_why_here(row):
    if row["reason_code"] == "STALE":
        return (
            f"Ranked here because it has {int(row['days_since_last_update'])} "
            f"days since update and {int(row['impressions_90d'])} impressions "
            f"in the 90-day window."
        )

    elif row["reason_code"] == "LOW_CTR_POSITION":
        return (
            f"Ranked here because CTR is {row['ctr']:.2f}% with "
            f"average position {row['avg_position']:.1f} and "
            f"{int(row['impressions_90d'])} impressions."
        )

    return (
        "Included because the baseline ranking places it among "
        "the highest-scoring rows."
    )


def get_what_could_make_it_wrong(row):
    if row["reason_code"] == "STALE":
        return (
            "It could be wrong if the page was recently reviewed or "
            "its current content is still appropriate despite the age."
        )

    elif row["reason_code"] == "LOW_CTR_POSITION":
        return (
            "It could be wrong if the low CTR is explained by "
            "query mix, SERP features, or other context not captured here."
        )

    return (
        "It could be wrong because the rule does not capture "
        "other page-level context."
    )


top10_review = top10[
    [
        "baseline_rank",
        "baseline_score",
        "reason_code",
        "action",
        "impressions_90d",
        "ctr",
        "avg_position",
        "days_since_last_update",
    ]
].copy()

top10_review["why_here"] = top10.apply(
    get_why_here,
    axis=1
).values

top10_review["confidence_note"] = top10.apply(
    get_confidence_note,
    axis=1
).values

top10_review["what_would_make_it_wrong"] = top10.apply(
    get_what_could_make_it_wrong,
    axis=1
).values


# ------------------------------------------------------------
# Print the required one-line review for each top-10 row
# ------------------------------------------------------------

print("=" * 80)
print("TOP-10 REVIEW")
print("=" * 80)

for _, row in top10_review.iterrows():

    print(
        f"\nRank {int(row['baseline_rank'])}: "
        f"Action={row['action']} | "
        f"Reason={row['reason_code']} | "
        f"Score={int(row['baseline_score'])}"
    )

    print(f"Why there: {row['why_here']}")

    print(
        f"Confidence: {row['confidence_note']}"
    )

    print(
        f"What would make it wrong: "
        f"{row['what_would_make_it_wrong']}"
    )

TOP-10 REVIEW

Rank 1: Action=REFRESH | Reason=STALE | Score=3
Why there: Ranked here because it has 194 days since update and 4556 impressions in the 90-day window.
Confidence: Higher confidence in rule fit: the page satisfies both observable conditions.
What would make it wrong: It could be wrong if the page was recently reviewed or its current content is still appropriate despite the age.

Rank 2: Action=REFRESH | Reason=STALE | Score=3
Why there: Ranked here because it has 301 days since update and 821 impressions in the 90-day window.
Confidence: Higher confidence in rule fit: the page satisfies both observable conditions.
What would make it wrong: It could be wrong if the page was recently reviewed or its current content is still appropriate despite the age.

Rank 3: Action=REFRESH | Reason=STALE | Score=3
Why there: Ranked here because it has 183 days since update and 545 impressions in the 90-day window.
Confidence: Higher confidence in rule fit: the page satisfies both observa

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks and leakage check

The weaker picks are pages where the rule identifies only one observable signal. These rows are useful for checking where the hand-written rule may be too broad.

A stale page may not need a refresh if its content is still appropriate or has been reviewed recently. A low-CTR page may also have another explanation, such as query mix, SERP features, or measurement noise.

I also checked the baseline score inputs for leakage. The score does not use `trend_direction`, `trend_pct`, or `is_declining_label`, and it does not use future-window measurements. These fields are not part of the ranking rule.

The resulting queue is therefore a transparent, directional decision-support list based on observable signals rather than a prediction of future performance.


In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# SECTION 4 — Weak picks + leakage check

print("=" * 80)
print("WEAK PICKS")
print("=" * 80)

# ------------------------------------------------------------
# Show a few lower-confidence score-1 picks
# ------------------------------------------------------------

weak_picks = ranked_queue[
    ranked_queue["baseline_score"] == 1
].head(5)

if len(weak_picks) == 0:

    print("No score-1 rows were found.")

else:

    print(
        weak_picks[
            [
                "baseline_rank",
                "baseline_score",
                "reason_code",
                "action",
                "impressions_90d",
                "ctr",
                "avg_position",
                "days_since_last_update",
            ]
        ].to_string(index=False)
    )


# ------------------------------------------------------------
# Leakage check
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("LEAKAGE CHECK")
print("=" * 80)

# These are the ONLY fields used to create the score.
score_inputs = {
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
}

# These are label/future-related fields that must NOT be
# used as scoring features.
forbidden_columns = {
    "is_declining_label",
    "trend_direction",
    "trend_pct",
}

future_window_columns = {
    "future_impressions",
    "future_clicks",
    "future_sessions",
    "next_30d_impressions",
    "next_30d_clicks",
    "next_30d_sessions",
}

print("Score inputs:")
for col in sorted(score_inputs):
    print(" -", col)

print("\nLabel-derived fields NOT used:")
for col in sorted(forbidden_columns):
    print(" -", col)

print("\nFuture-window fields NOT used:")
for col in sorted(future_window_columns):
    print(" -", col)


# ------------------------------------------------------------
# Check the actual score formula
# ------------------------------------------------------------

score_formula_inputs = {
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
}

label_leakage = score_formula_inputs.intersection(
    forbidden_columns
)

future_leakage = score_formula_inputs.intersection(
    future_window_columns
)

print("\n" + "-" * 80)

if len(label_leakage) == 0:
    print("PASS: No label-derived fields are used in the score.")
else:
    print(
        "CHECK: Label-derived fields found:",
        label_leakage
    )

if len(future_leakage) == 0:
    print("PASS: No future-window fields are used in the score.")
else:
    print(
        "CHECK: Future-window fields found:",
        future_leakage
    )

print("\nFinal leakage status:")

if len(label_leakage) == 0 and len(future_leakage) == 0:
    print("PASS — baseline score uses only observable non-label inputs.")
else:
    print("CHECK REQUIRED — review the score inputs.")

print("\nLabel used in score formula: NO")
print("Future-window data used in score formula: NO")

WEAK PICKS
 baseline_rank  baseline_score      reason_code     action  impressions_90d  ctr  avg_position  days_since_last_update
            18               1 LOW_CTR_POSITION CTR_REVIEW            11751 0.49           6.2                      22
            19               1 LOW_CTR_POSITION CTR_REVIEW             3970 0.03           8.5                      20
            20               1 LOW_CTR_POSITION CTR_REVIEW             1240 0.16           4.9                     104
            21               1 LOW_CTR_POSITION CTR_REVIEW            13848 0.15           8.9                     104
            22               1 LOW_CTR_POSITION CTR_REVIEW             9449 0.07           7.3                      22

LEAKAGE CHECK
Score inputs:
 - avg_position
 - ctr
 - days_since_last_update
 - impressions_90d

Label-derived fields NOT used:
 - is_declining_label
 - trend_direction
 - trend_pct

Future-window fields NOT used:
 - future_clicks
 - future_impressions
 - future_sessions
 -

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.